## Module 1: Agentic RAG

This module consists of building a simple RAG pipeline using keyword search, and making it agentic, so the LLM decides when and what to search instead of running a fixed pipeline.

### Part 1: RAG

RAG (*Retrieval-Augmented Generation*) is a technique used to improve the accuracy and reliability of LLMs (*Large Language Models*) by fetching facts from an external knowledge base before generating a response. In such cases, where an external knowledge base is the context, usual LLMs can only give generic answers without having access to that specific knowledge. 

In [6]:
# make sure .env file is loaded
from dotenv import load_dotenv
print(load_dotenv())

# check whether the OpenAI client works
from openai import OpenAI
openai_client = OpenAI()

True


In [7]:
# define a function to talk to llm and test it
def llm(prompt):
    response = openai_client.responses.create(
        model = 'gpt-5.4-mini',
        input = prompt
    )
    return response.output_text

llm("Hey, what's up?")

'Hey! Not much — just here and ready to help. What’s going on?'

Here we are asking an external knowledge base specific question without providing th context:

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

Maybe — it depends on the course’s start date and whether enrollment is still open.

If you want, I can help you draft a quick message to the instructor or course organizer, for example:

> Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll at this point? If so, please let me know the next steps. Thank you!

If you tell me the course name or whether you’re asking as a student or for a specific platform, I can make this more precise.


The model returns a very generic answer since it has no knowledge about that specific course. Here we provide the model some context to depend on. The prompt doesn't end with "Answer:". With older models like GPT-3 it was added to nudge the model into completing the sentence. Modern models don't need the hint.

In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [10]:
question = "I just discovered the course. Can I join now?"
answer = llm(prompt)
print(answer)

Yes, you can still join now. If you want to receive a certificate, make sure to submit your project while submissions are still open.


Generation in RAG is the LLM producing the text, and retrieval is the search. Relevant documents are retrieved from the knowledge base and used to augment what the LLM generates. That search step is what gives the LLM the context it needs to answer correctly. What was done above was naive, since FAQ entry already held the answer and it was pasted as context by hand. What the actual RAG does instead is to perform search automatically. Then LLM sees only the documents handed to it, so the model is only as good as the retrieved data.

In [1]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [2]:
import requests

# get courses data from the faq page 
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

documents = []
url_prefix = "https://datatalks.club/faq"

# get q&a pairs for each course and add to documents
for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [4]:
# check the courses
courses_raw

[{'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'stock-markets-analytics-zoomcamp',
  'course_name': 'Stock Markets Analytics Zoomcamp',
  'path': '/json/stock-markets-analytics-zoomcamp.json',
  'questions_count': 93},
 {'course': 'ai-dev-tools-zoomcamp',
  'course_name': 'AI Dev Tools Zoomcamp',
  'path': '/json/ai-dev-tools-zoomcamp.json',
  'questions_count': 41},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255},
 {'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472}]

In [14]:
# check a sample q&a
documents[600]

{'id': 'eae0fb50aa',
 'course': 'llm-zoomcamp',
 'section': 'Capstone Project',
 'question': 'When and how will we be assigned projects for review/grading?',
 'answer': 'After the submission deadline has passed, an Excel sheet will be shared with 3 projects being assigned to each participant.'}